# 07 — Profile Registry & Fidelity Validation

Save named profiles from reference data, then validate freshly generated data matches them using KS/Chi-squared tests with an HTML report.

In [ ]:
from sqllocks_spindle import Spindle, ProfileRegistry
from sqllocks_spindle.inference import DataProfiler, DatasetProfile
import importlib
from sqllocks_spindle.cli import _get_domain_registry

# Generate reference data from the retail domain
spindle = Spindle()
reg_info = _get_domain_registry()
mod, cls, _ = reg_info['retail']
domain = getattr(importlib.import_module(mod), cls)(schema_mode='3nf')
ref_result = spindle.generate(domain=domain, scale='small', seed=42)
print(ref_result)

In [ ]:
# Profile the reference tables and save to registry
from pathlib import Path
profiler = DataProfiler(sample_rows=500)
reg = ProfileRegistry(root=Path('/tmp/spindle_profiles'))

table_profiles = {
    name: profiler.profile(df, table_name=name)
    for name, df in ref_result.tables.items()
}
dataset_profile = DatasetProfile(tables=table_profiles)
saved = reg.save_from_dataset_profile(dataset_profile, system='retail_demo', name='baseline-2026')
print(f'Saved {len(saved)} profiles')
for p in saved:
    print(f'  {p.identity}  ({p.source_rows} rows)')

In [ ]:
# Generate fresh data and run fidelity validation
new_result = spindle.generate(domain=domain, scale='small', seed=99)

# Validate each saved profile
for profile in saved[:3]:  # first 3 tables
    try:
        report = reg.validate(profile.identity, new_result)
        print(f'\n{profile.identity}: {report.overall_score:.1f}/100')
        print(report.summary()[:400])
    except KeyError:
        pass  # table not in result (different domain scale)

In [ ]:
# Export HTML report for the first profile
try:
    report = reg.validate(saved[0].identity, new_result)
    html = report.to_html(title=f'Fidelity: {saved[0].identity}')
    Path('/tmp/fidelity_report.html').write_text(html)
    print(f'HTML report written ({len(html):,} bytes)')
except Exception as e:
    print(f'Report skipped: {e}')

In [ ]:
# List registry contents
entries = reg.list_all()
print(f'Registry has {len(entries)} profiles')
for e in entries[:5]:
    print(f'  {e["system"]}/{e["table"]}/{e["name"]}')